In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [39]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

Existing Spark session found. Stopping it...


In [40]:
spark = (
        SparkSession.builder
        .appName("Adaptive Query Execution App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

In [41]:
spark

In [42]:
spark.conf.set("spark.sql.adaptive.enabled", "true") # Enable Adaptive Query Execution

### spark.sql.adaptive.coalescePartitions.enabled

In [43]:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

##### When enabled, Spark dynamically reduces the number of shuffle partitions at runtime based on the actual size of data. Normally, spark.sql.shuffle.partitions is fixed (default = 200). If your data is much smaller or larger than expected, the number of partitions may be inefficient (too many tiny files or too few huge partitions).

##### df.repartition(N) is a pre-shuffle hint: it tells Spark how many partitions to use before a shuffle.
##### With AQE + coalescePartitions, Spark reoptimizes the shuffle stage after execution stats are collected.
##### If Spark finds that some partitions are too small, it will coalesce them (merge into fewer partitions).
##### So the final partition count after shuffle may be different from what you requested.

In [44]:
df = spark.range(0, 1000).repartition(50)

In [45]:
result = df.groupBy("id").count()

In [46]:
result.rdd.getNumPartitions()

[Stage 2:=========================================>              (37 + 13) / 50]

1

In [47]:
df.rdd.getNumPartitions()

50

## spark.sql.adaptive.skewJoin.enabled

##### This handles data skew during joins. If one partition is much larger than others, Spark splits that skewed partition into smaller chunks and redistributes them, so no single task becomes a bottleneck.

In [48]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

In [49]:
df1 = spark.range(0, 1000).withColumnRenamed("id", "key")
df2 = spark.range(0, 10000).withColumn("key", (col("id") % 2))

In [50]:
joined = df1.join(df2, on="key")

In [51]:
df1.rdd.getNumPartitions()

12

In [52]:
df2.rdd.getNumPartitions()

12

In [53]:
joined.rdd.getNumPartitions()

12

In [54]:
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [key])
:- Project [id#924L AS key#926L]
:  +- Range (0, 1000, step=1, splits=Some(12))
+- Project [id#928L, (id#928L % cast(2 as bigint)) AS key#930L]
   +- Range (0, 10000, step=1, splits=Some(12))

== Analyzed Logical Plan ==
key: bigint, id: bigint
Project [key#926L, id#928L]
+- Join Inner, (key#926L = key#930L)
   :- Project [id#924L AS key#926L]
   :  +- Range (0, 1000, step=1, splits=Some(12))
   +- Project [id#928L, (id#928L % cast(2 as bigint)) AS key#930L]
      +- Range (0, 10000, step=1, splits=Some(12))

== Optimized Logical Plan ==
Project [key#926L, id#928L]
+- Join Inner, (key#926L = key#930L)
   :- Project [id#924L AS key#926L]
   :  +- Range (0, 1000, step=1, splits=Some(12))
   +- Project [id#928L, (id#928L % 2) AS key#930L]
      +- Filter isnotnull((id#928L % 2))
         +- Range (0, 10000, step=1, splits=Some(12))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(2) Project [key#926L, 

## spark.sql.adaptive.join.enabled

##### This lets Spark change the join strategy at runtime (e.g., switch from Sort-Merge Join → Broadcast Hash Join) if actual data sizes are smaller/larger than expected.

In [55]:
spark.conf.set("spark.sql.adaptive.join.enabled", "true")

In [56]:
small_df = spark.range(0, 100).withColumnRenamed("id", "key")

In [57]:
large_df = spark.range(0, 10000000).withColumn("key", (col("id") % 2))

In [58]:
joined = small_df.join(large_df, "key")

In [59]:
joined.explain(False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#937L, id#939L]
   +- BroadcastHashJoin [key#937L], [key#941L], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=1166]
      :  +- Project [id#935L AS key#937L]
      :     +- Range (0, 100, step=1, splits=12)
      +- Project [id#939L, (id#939L % 2) AS key#941L]
         +- Filter isnotnull((id#939L % 2))
            +- Range (0, 10000000, step=1, splits=12)




In [60]:
joined.explain(mode="cost") # mode = formatted, codegen, cost

== Optimized Logical Plan ==
Project [key#937L, id#939L], Statistics(sizeInBytes=67.1 GiB)
+- Join Inner, (key#937L = key#941L), Statistics(sizeInBytes=89.4 GiB)
   :- Project [id#935L AS key#937L], Statistics(sizeInBytes=800.0 B)
   :  +- Range (0, 100, step=1, splits=Some(12)), Statistics(sizeInBytes=800.0 B, rowCount=100)
   +- Project [id#939L, (id#939L % 2) AS key#941L], Statistics(sizeInBytes=114.4 MiB)
      +- Filter isnotnull((id#939L % 2)), Statistics(sizeInBytes=76.3 MiB)
         +- Range (0, 10000000, step=1, splits=Some(12)), Statistics(sizeInBytes=76.3 MiB, rowCount=1.00E+7)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#937L, id#939L]
   +- BroadcastHashJoin [key#937L], [key#941L], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=1166]
      :  +- Project [id#935L AS key#937L]
      :     +- Range (0, 100, step=1, splits=12)
      +- Project [id#939L, (id#939L % 2) AS 

In [61]:
spark.stop()